In [1]:
!pip install -q -U langchain langchain-community langchain-google-genai chromadb pypdf wandb python-dotenv
print("Installation complete. The 'ERROR' messages above are typically dependency conflicts with pre-installed Colab packages and usually do not affect the functionality of the libraries you just installed (langchain, etc.). You can often ignore them.")

Installation complete. The 'ERROR' messages above are typically dependency conflicts with pre-installed Colab packages and usually do not affect the functionality of the libraries you just installed (langchain, etc.). You can often ignore them.


In [2]:
import os
from google.colab import userdata

# Mengambil API Key dari Google Colab Secrets
try:
    gemini_key = userdata.get('GEMINI_API_KEY')
    wandb_key = userdata.get('WANDB_API_KEY')
    if gemini_key and wandb_key:
        print("✅ API Keys berhasil dimuat dengan aman!")
    else:
        raise userdata.SecretNotFoundError("One or more API keys are missing.")
except userdata.SecretNotFoundError as e:
    print(f"❌ Error: {e}. Please ensure you have set the API keys in Colab Secrets.")
    print("To set secrets: Click the '🔑' icon on the left sidebar (or go to Runtime -> View secrets), and add your 'GEMINI_API_KEY' and 'WANDB_API_KEY'.")
    print("After adding the secrets, run this cell again.")

✅ API Keys berhasil dimuat dengan aman!


In [3]:
import wandb

# Inisialisasi Project W&B
# Ubah nama entity (username W&B) dan project sesuai akunmu
wandb.init(
    project="rag-pdf-chatbot-week11",
    entity="xxenonitee-stikomelrahma", # Updated based on your W&B login info. Please verify.
    name="baseline_vs_rag_experiment"
)

# Menyiapkan W&B Table untuk mencatat log hasil eksperimen
columns = ["Query", "Baseline (Tanpa RAG)", "RAG (Dengan RAG)", "Source Citation"]
wandb_table = wandb.Table(columns=columns)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: xxenonitee (xxenonitee-stikomelrahma) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [4]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Ganti ke nama file PDF yang sudah kita buat otomatis di sel sebelumnya
pdf_path = "panduan_fotografi_expert.pdf"

# 2. Memuat dokumen menggunakan PyPDFLoader
loader = PyPDFLoader(pdf_path)
docs = loader.load()

# 3. Melakukan Text Splitting (Chunking) agar teks menjadi potongan kecil
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_documents(docs)

print(f"📄 Sukses memuat file: {pdf_path}")
print(f"Total Halaman PDF: {len(docs)}")
print(f"✂️ Total Potongan Teks (Chunks): {len(chunks)}")

/tmp/ipykernel_23251/196330266.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


📄 Sukses memuat file: panduan_fotografi_expert.pdf
Total Halaman PDF: 1
✂️ Total Potongan Teks (Chunks): 3


In [5]:
# 1. Install library untuk membuat PDF
!pip install -q reportlab

from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle

def create_photography_pdf():
    pdf_filename = "panduan_fotografi_expert.pdf"
    doc = SimpleDocTemplate(pdf_filename, pagesize=letter,
                            rightMargin=54, leftMargin=54, topMargin=54, bottomMargin=54)
    story = []

    styles = getSampleStyleSheet()

    # Custom Styles
    title_style = ParagraphStyle(
        'TitleStyle',
        parent=styles['Heading1'],
        fontSize=22,
        leading=26,
        spaceAfter=20
    )
    heading_style = ParagraphStyle(
        'HeadingStyle',
        parent=styles['Heading2'],
        fontSize=14,
        leading=18,
        spaceBefore=12,
        spaceAfter=6
    )
    body_style = ParagraphStyle(
        'BodyStyle',
        parent=styles['BodyText'],
        fontSize=10,
        leading=14,
        spaceAfter=8
    )

    # Isi Konten Dokumen PDF
    story.append(Paragraph("<b>PANDUAN TEKNIS FOTOGRAFI EXPERT: METODE NEOCLASSIC LIGHTING (2026)</b>", title_style))
    story.append(Spacer(1, 12))

    story.append(Paragraph("<b>1. Pengenalan Metode Neoclassic Lighting</b>", heading_style))
    story.append(Paragraph(
        "Metode Neoclassic Lighting adalah teknik pencahayaan studio modern yang menggabungkan estetika lukisan "
        "Renaissance dengan teknologi sensor kamera mirrorless resolusi tinggi. Kunci dari metode ini adalah "
        "menciptakan gradasi bayangan yang sangat halus (soft fall-off) pada wajah subjek, memberikan kesan dimensi "
        "tiga dimensi yang dramatis namun tetap mempertahankan detail tekstur kulit tanpa digital retouching berlebih.",
        body_style
    ))

    story.append(Paragraph("<b>2. Formula Segitiga Eksposur Khusus (The Golden Ratio Exposure)</b>", heading_style))
    story.append(Paragraph(
        "Untuk menghasilkan foto Neoclassic yang sempurna pada kondisi studio tertutup, fotografer wajib menggunakan "
        "rumus pengaturan eksposur spesifik berikut: Aperture harus diatur tepat pada f/2.8 untuk memisahkan subjek dari "
        "latar belakang secara lembut. Shutter speed wajib dikunci pada 1/160 detik untuk menghindari adanya sinkronisasi "
        "rana yang meleset dengan lampu studio (flash sync). Terakhir, tingkat ISO harus dijaga pada nilai ISO 64 guna "
        "mendapatkan dynamic range tertinggi dan menekan noise seminimal mungkin pada area bayangan.",
        body_style
    ))

    story.append(Paragraph("<b>3. Konfigurasi Lampu dan Modifier</b>", heading_style))
    story.append(Paragraph(
        "Sistem ini menggunakan setup 3 titik lampu dengan aturan penempatan vertikal dan sudut yang sangat ketat. "
        "Lampu Utama (Key Light) menggunakan Octabox berukuran 120cm yang diletakkan di sisi kanan subjek dengan sudut "
        "tepat 45 derajat secara horizontal dan diarahkan menunduk 30 derajat ke bawah. Lampu kedua adalah Fill Light "
        "menggunakan Umbrella reflektif putih berukuran 90cm di sisi kiri kamera dengan kekuatan 1/4 dari Key Light. "
        "Lampu ketiga, Rim Light, ditempatkan di belakang subjek sebelah kiri dengan ketinggian 2 meter, diarahkan ke rambut "
        "untuk memberikan efek pemisahan (separation) dari background hitam.",
        body_style
    ))

    story.append(Paragraph("<b>4. Aturan Komposisi 'The Fibonacci Inverse'</b>", heading_style))
    story.append(Paragraph(
        "Berbeda dengan aturan sepertiga (Rule of Thirds) konvensional, panduan ini mewajibkan penggunaan teknik "
        "Fibonacci Inverse. Mata kanan subjek harus diposisikan tepat pada titik spiral terdalam dari rasio emas yang "
        "dibalik secara vertikal. Teknik ini memaksa pandangan audiens untuk menjelajahi area gelap (shadow) terlebih "
        "dahulu sebelum akhirnya mendarat pada bagian wajah yang terang (highlight).",
        body_style
    ))

    doc.build(story)
    print(f"✅ Berhasil membuat file PDF: {pdf_filename}")

create_photography_pdf()

✅ Berhasil membuat file PDF: panduan_fotografi_expert.pdf


In [6]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = "panduan_fotografi_expert.pdf" # Masukkan nama PDF baru ini
loader = PyPDFLoader(pdf_path)
docs = loader.load()
print(f"✅ PDF loaded successfully: {len(docs)} pages.")

✅ PDF loaded successfully: 1 pages.


In [7]:
query = "Berapa pengaturan Aperture, Shutter Speed, dan ISO yang wajib digunakan pada formula eksposur Neoclassic Lighting?"

In [8]:
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma
from google.colab import userdata # Import userdata

# 1. Melakukan Text Splitting pada dokumen PDF fotografi yang sudah di-load
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)
chunks = text_splitter.split_documents(docs)
print(f"✂️ Total potongan teks (chunks): {len(chunks)}")

# 2. Inisialisasi Model Embedding Google
google_api_key = userdata.get('GEMINI_API_KEY') # Directly get from userdata

if not google_api_key:
    print("❌ Error: GOOGLE_API_KEY environment variable is not set.")
    print("Please ensure you have run the cell (HEWp2LfqCuf5) to load API keys from Colab Secrets.")
    print("If you have, verify that 'GEMINI_API_KEY' is correctly set in Colab Secrets and has a valid value.")
    raise ValueError("GOOGLE_API_KEY is missing.")
else:
    print("✅ GOOGLE_API_KEY found in environment variables. Initializing embeddings...")
    embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001", google_api_key=google_api_key)

    # 3. Simpan potongan teks ke dalam ChromaDB
    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings
    )

    # 4. Ambil top-k dokumen teratas yang paling relevan
    retriever = vector_store.as_retriever(search_kwargs={"k": 3})
    print("🗄️ Vector Database (ChromaDB) berhasil dibuat!")

✂️ Total potongan teks (chunks): 3
✅ GOOGLE_API_KEY found in environment variables. Initializing embeddings...
🗄️ Vector Database (ChromaDB) berhasil dibuat!


In [9]:
import google.generativeai as genai
from google.colab import userdata

# Ensure the GOOGLE_API_KEY is set by getting it directly
google_api_key = userdata.get('GEMINI_API_KEY')

if not google_api_key:
    print("❌ Error: GEMINI_API_KEY not found in Colab Secrets. Cannot list models.")
    print("Please ensure you have run the API key loading cell (HEWp2LfqCuf5) and 'GEMINI_API_KEY' is correctly set.")
else:
    genai.configure(api_key=google_api_key)
    print("✅ GOOGLE_API_KEY found. Listing available generative models (supporting 'generateContent')...")

    found_generative_models = False
    for m in genai.list_models():
        if "generateContent" in m.supported_generation_methods:
            print(f"  - {m.name}")
            found_generative_models = True
    if not found_generative_models:
        print("No generative models supporting 'generateContent' were found for your API key.")
    print("Please select one of the available generative models for ChatGoogleGenerativeAI.")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


✅ GOOGLE_API_KEY found. Listing available generative models (supporting 'generateContent')...
  - models/gemini-2.5-flash
  - models/gemini-2.5-pro
  - models/gemini-2.0-flash
  - models/gemini-2.0-flash-001
  - models/gemini-2.0-flash-lite-001
  - models/gemini-2.0-flash-lite
  - models/gemini-2.5-flash-preview-tts
  - models/gemini-2.5-pro-preview-tts
  - models/gemma-4-26b-a4b-it
  - models/gemma-4-31b-it
  - models/gemini-flash-latest
  - models/gemini-flash-lite-latest
  - models/gemini-pro-latest
  - models/gemini-2.5-flash-lite
  - models/gemini-2.5-flash-image
  - models/gemini-3-pro-preview
  - models/gemini-3-flash-preview
  - models/gemini-3.1-pro-preview
  - models/gemini-3.1-pro-preview-customtools
  - models/gemini-3.1-flash-lite-preview
  - models/gemini-3.1-flash-lite
  - models/gemini-3-pro-image-preview
  - models/gemini-3-pro-image
  - models/nano-banana-pro-preview
  - models/gemini-3.1-flash-image-preview
  - models/gemini-3.1-flash-image
  - models/gemini-3.5-flas

In [10]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from google.colab import userdata # Import userdata

# Retrieve API key directly from userdata
google_api_key = userdata.get('GEMINI_API_KEY')
if not google_api_key:
    raise ValueError("GOOGLE_API_KEY not found. Please ensure it's set in Colab Secrets and the API key loading cell is run.")

# Inisialisasi Model Gemini-Flash-Latest, passing the API key explicitly
llm = ChatGoogleGenerativeAI(model="gemini-flash-latest", temperature=0.3, google_api_key=google_api_key)

# Membuat Prompt Template khusus RAG
system_prompt = (
    "Anda adalah asisten ahli fotografi. Gunakan potongan konteks berikut untuk menjawab "
    "pertanyaan di akhir. Jika Anda tidak tahu jawabannya, katakan bahwa Anda tidak tahu.\n\n"
    "Konteks:\n{context}\n\n"
    "Pertanyaan: {input}"
)
prompt_template = ChatPromptTemplate.from_template(system_prompt)

# Fungsi pembantu menggabungkan potongan dokumen PDF
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Membangun RAG Chain dengan LCEL Pipe (|) - AMAN DARI ERROR MODUL
rag_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

print("🤖 RAG Chain siap mengeksekusi pertanyaan!")

🤖 RAG Chain siap mengeksekusi pertanyaan!


In [11]:
query = "Berapa pengaturan Aperture, Shutter Speed, dan ISO yang wajib digunakan pada formula eksposur Neoclassic Lighting?"

# 1. Jalankan Baseline
print("⏳ Menghasilkan jawaban Baseline...")
baseline_text = llm.invoke(query).content

# 2. Jalankan RAG
print("⏳ Menghasilkan jawaban RAG...")
rag_text = rag_chain.invoke(query)

# Tampilkan Hasil di Layar
print("\n" + "="*50)
print(f"❓ PERTANYAAN:\n{query}\n")
print(f"❌ JAWABAN BASELINE:\n{baseline_text}\n")
print(f"✅ JAWABAN RAG:\n{rag_text}\n")
print("="*50 + "\n")

# 3. Kirim ke W&B
wandb_table.add_data(query, baseline_text, rag_text, "Halaman 1")
wandb.log({"Hasil_Perbandingan_RAG": wandb_table})
wandb.finish()
print("🚀 Selesai! Data sukses terkirim ke W&B.")

⏳ Menghasilkan jawaban Baseline...


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [ ]:
query = "Berapa pengaturan Aperture, Shutter Speed, dan ISO yang wajib digunakan pada formula eksposur Neoclassic Lighting?"

# 1. Jalankan Baseline
print("⏳ Menghasilkan jawaban Baseline...")
baseline_text = llm.invoke(query).content

# 2. Jalankan RAG
print("⏳ Menghasilkan jawaban RAG...")
rag_text = rag_chain.invoke(query)

# Tampilkan Hasil di Layar
print("\n" + "="*50)
print(f"❓ PERTANYAAN:\n{query}\n")
print(f"❌ JAWABAN BASELINE:\n{baseline_text}\n")
print(f"✅ JAWABAN RAG:\n{rag_text}\n")
print("="*50 + "\n")

# 3. Kirim ke W&B
# Ensure a W&B run is active before logging. If wandb.finish() was called previously,
# you might need to re-run the `wandb.init()` cell (r2JK1aXUDOYK) to start a new run.
# We are removing wandb.finish() from here to allow continuous logging if desired.
wandb_table.add_data(query, baseline_text, rag_text, "Halaman 1")
wandb.log({"Hasil_Perbandingan_RAG": wandb_table})
# wandb.finish() # Removed to prevent premature ending of the W&B run
print("🚀 Data berhasil dikirim ke W&B (run masih aktif).")